### LLM에게 도구 붙여주기
- LLM은 기본적으로 질문하면 답변만 하는 모델
- 우리가 만든 함수(도구)를 골라 쓸 수 있도록 해준다
- AI 에이전트의 핵심 - RAG도 마찬가지

- LLM에게 얘기하는 방식
    - 너에게 ~~한 도구가 있다.(날씨 조회, 검색 등)
    - LLM이 필요로 할 때 알아서 도구 선택해서 실행하는 방식이 AI 에이전트.
    - LLM이 스스로 판단해서 도구를 쓰고 결과도 확인해서 최종 결과 두는 방식.

- 작동 방식
    - 함수 실행은 우리가 하고, 그 결과를 다시 LLM이 받아서 최종 자연스러운 결과를 우리에게 주는 방식

#### 전체 흐름
1. LLM에게 사용 가능한 tool 목록을 알려줍니다.
2. LLM이 필요할 때 알아서 이 함수를 실행하라고 요청합니다.
3. 우리가 그 함수를 실행합니다.
4. 함수 실행 결과를 LLM에게 돌려주면 LLM이 최종 답을 만들어 반환합니다.

즉 LLM은 판단하고 우리는 실행!

In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()

In [2]:
# 임시로 함수를 만들어 붙여보기

def get_weather(city):
    # 실제로는 날씨 웹검색이 들어가야하지만, 일단 임시로 고정값 넣음
    return f"{city}의 날씨는 맑음이고, 기온이 50도"

print(get_weather("신대방"))

신대방의 날씨는 맑음이고, 기온이 50도


### 1. 도구를 LLM에게 설명하기
- LLM에게 우리가 만든 이 함수를 설명해줘야
    - 이름, 설명, 인자 등
    - tools 에게 자세한 명세를 적어야 한다.'
    - 함수 이름은 영문으로, 설명은 LLM이 이해할 수 있도록

In [3]:
tools = [{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "특정 도시의 현재 날씨를 조회한다",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "날씨를 알고 싶은 도시 이름"},
            },
            "required": ["city"],
        },
    },
}]
print("도구 정의 완료")

도구 정의 완료


### 2. LLM에게 도구 호출 결정권을 준다.

In [36]:
messages = [

    {"role" : "user", "content" : "신대방 날씨 어때?"}
]

response = client.chat.completions.create(
    model="gpt-5.6-luna",
    tools=tools,
    reasoning_effort="none",
    messages=messages
)

In [38]:
# call = response.tool_calls[0]
call = response.choices[0].message.tool_calls[0]

In [39]:
print(call.function.name)
print(call.function.arguments)

get_weather
{"city":"신대방"}


### 3. 우리가 함수 실제로 실행

In [40]:
args = call.function.arguments
type(args)

str

In [41]:
import json
args = json.loads(call.function.arguments)

In [42]:
tool_result = get_weather(**args)
print(tool_result)

신대방의 날씨는 맑음이고, 기온이 50도


In [43]:
response

ChatCompletion(id='chatcmpl-EOdAJKRYBwtQ3ytJpOv0iIEEeraVB', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_Nve7YqRRlTi3ziSnCOUq6IzF', function=Function(arguments='{"city":"신대방"}', name='get_weather'), type='function')]))], created=1789539131, model='gpt-5.6-luna', object='chat.completion', metadata=None, moderation=None, service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=19, prompt_tokens=145, total_tokens=164, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0, text_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=0, cached_tokens=0, image_tokens=None, text_tokens=None)))

### 4. 결과를 LLM에게 돌려서 최종 답 만들기

In [44]:
messages.append(response.choices[0].message)

In [46]:
messages.append({"role" : "tool", "tool_call_id" : call.id, "content" : tool_result})
messages

[{'role': 'user', 'content': '신대방 날씨 어때?'},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_Nve7YqRRlTi3ziSnCOUq6IzF', function=Function(arguments='{"city":"신대방"}', name='get_weather'), type='function')]),
 {'role': 'tool',
  'tool_call_id': 'call_Nve7YqRRlTi3ziSnCOUq6IzF',
  'content': '신대방의 날씨는 맑음이고, 기온이 50도'}]

In [47]:
# 1. 질문
response1 = client.chat.completions.create(
    model="gpt-5.6-luna",
    tools=tools,
    reasoning_effort="none",
    messages=messages
)

In [48]:
print(response1.choices[0].message.content)

신대방은 현재 **맑고, 기온은 50도**예요.


In [49]:
# 함수로 만들어서 사용할 수 있음
available_tools = {"get_weather": get_weather}

def chat_with_tools(question, tools):
    messages = [{"role": "user", "content": question}]
    r = client.chat.completions.create(model="gpt-5.6-luna", tools=tools,
                                       reasoning_effort="none", messages=messages)
    calls = r.choices[0].message.tool_calls
    if not calls:                       # 도구가 필요 없으면 바로 답
        return r.choices[0].message.content
    messages.append(r.choices[0].message)
    for tc in calls:                    # 필요한 도구를 모두 실행
        args = json.loads(tc.function.arguments)
        result = available_tools[tc.function.name](**args)
        messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
    r2 = client.chat.completions.create(model="gpt-5.6-luna", tools=tools,
                                        reasoning_effort="none", messages=messages)
    return r2.choices[0].message.content

print(chat_with_tools("부산 날씨 알려줘", tools))

부산은 현재 **맑고, 기온은 50도**입니다.


In [54]:
# 다른 툴 추가

def recommend_clothes(data):
    return f"{data}의 스타일은 너무 멋져요!"

tools = [{
    "type": "function",
    "function": {
        "name": "recommend_clothes",
        "description": "옷에 대해서 물어봤을 때 코멘트를 해준다. 80년대 스타일이라는 말 꼭 넣는다",
        "parameters": {
            "type": "object",
            "properties": {
                "data": {"type": "string", "description": "코멘트 받고 싶은 옷 이름"},
            },
            "required": ["data"],
        },
    },
},
{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "특정 도시의 현재 날씨를 조회한다",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "날씨를 알고 싶은 도시 이름"},
            },
            "required": ["city"],
        },
    },
}
]

In [55]:
# 함수로 만들어서 사용할 수 있음
available_tools = {"get_weather": get_weather, "recommend_clothes" : recommend_clothes}

def chat_with_tools(question, tools):
    messages = [{"role": "user", "content": question}]
    r = client.chat.completions.create(model="gpt-5.6-luna", tools=tools,
                                       reasoning_effort="none", messages=messages)
    calls = r.choices[0].message.tool_calls
    if not calls:                       # 도구가 필요 없으면 바로 답
        return r.choices[0].message.content
    messages.append(r.choices[0].message)
    for tc in calls:                    # 필요한 도구를 모두 실행
        args = json.loads(tc.function.arguments)
        result = available_tools[tc.function.name](**args)
        messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
    r2 = client.chat.completions.create(model="gpt-5.6-luna", tools=tools,
                                        reasoning_effort="none", messages=messages)
    return r2.choices[0].message.content

print(chat_with_tools("이 가죽 자켓 어때?", tools))

가죽 자켓 멋져요! 시크하고 활용도가 높으며, 청바지나 티셔츠와 매치하면 세련된 **80년대 스타일** 분위기를 낼 수 있어요.


#### 데이터프레임에서 뉴스 내용 조회해서 요약해주기
- 데이터프레임에서 제일 첫번째 내용만 요약해주는 도구

In [56]:
import pandas as pd

df = pd.read_csv("../data/11-1_뉴스정제.csv")
df.head(3)

,제목,본문,카테고리,요약,출처URL,정제본문
0,현대백화점그룹 더현대 광주 추진,서울 연합뉴스 현대백화점그룹이 광주광역시에 서울 여의도 더현대 서울 과 같은 문화복...,경제,"6 6일 현대백화점그룹이 광주시에 문화복합몰을 만든다고 6일 밝혔으며, 광주시는 서...",https://n.news.naver.com/mnews/article/001/001...,서울 연합뉴스 현대백화점그룹이 광주광역시에 서울 여의도 더현대 서울 과 같은 문화복...
1,이스타항공 이상직 회사와 무관…오해 살 언동 말아야,전주 뉴시스 김얼 기자 이스타항공 자금 배임·횡령으로 전주교도소에 수감됐었던 이상직...,경제,이이스항공은 자금 배임·횡령으로 전주교도소에 수감됐었던 이상직 전 의원이 출소한 것...,https://n.news.naver.com/mnews/article/003/001...,전주 뉴시스 김얼 기자 이스타항공 자금 배임 횡령으로 전주교도소에 수감됐었던 이상직...
2,농협은행 농협금융 출범 10주년 기념주화 NFT 이벤트,NH농협은행은 올해 농협금융 출범 10주년을 맞아 이달 29일까지 ‘10주년 기념주...,경제,NH농협은행은 올해 농협금융 출범 10주년을 맞아 이달 29일까지 소셜미디어 인스타...,https://n.news.naver.com/mnews/article/366/000...,NH농협은행은 올해 농협금융 출범 10주년을 맞아 이달 29일까지 10주년 기념주화...


In [58]:
# 다른 툴 추가하기 - csv에서 제목이나 본문 물어볼 수 있음

def check_data(column):  # 본문 물어보면 본문, 제목 물어보면 제목 도출
    df = pd.read_csv("../data/11-1_뉴스정제.csv")
    return df[column][0]

tools = [{
    "type": "function",
    "function": {
        "name": "recommend_clothes",
        "description": "옷에 대해서 물어봤을 때 코멘트를 해준다. 80년대 스타일이라는 말 꼭 넣는다",
        "parameters": {
            "type": "object",
            "properties": {
                "data": {"type": "string", "description": "코멘트 받고 싶은 옷 이름"},
            },
            "required": ["data"],
        },
    },
},
{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "특정 도시의 현재 날씨를 조회한다",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "날씨를 알고 싶은 도시 이름"},
            },
            "required": ["city"],
        },
    },
},
{
    "type": "function",
    "function": {
        "name": "check_data",
        "description": "데이터프레임에서 내용 조회할 때 칼럼명으로 조회해서 알려준다",
        "parameters": {
            "type": "object",
            "properties": {
                "column": {"type": "string", "description": "조회하고 싶은 칼럼"},
            },
            "required": ["column"],
        },
    },
}
]

In [62]:
# 함수로 만들어서 사용할 수 있음
available_tools = {"get_weather": get_weather, 
                   "recommend_clothes" : recommend_clothes,
                   "check_data" : check_data}

def chat_with_tools(question, tools):
    messages = [{"role": "user", "content": question}]
    r = client.chat.completions.create(model="gpt-5.6-luna", tools=tools,
                                       reasoning_effort="none", messages=messages)
    calls = r.choices[0].message.tool_calls
    if not calls:                       # 도구가 필요 없으면 바로 답
        return r.choices[0].message.content
    messages.append(r.choices[0].message)
    for tc in calls:                    # 필요한 도구를 모두 실행
        args = json.loads(tc.function.arguments)
        result = available_tools[tc.function.name](**args)
        messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
    r2 = client.chat.completions.create(model="gpt-5.6-luna", tools=tools,
                                        reasoning_effort="none", messages=messages)
    return r2.choices[0].message.content

print(chat_with_tools("데이터프레임에서 본문 알려줘", tools))

본문: 서울 연합뉴스 현대백화점그룹이 광주광역시에 서울 여의도 더현대 서울과 같은 문화복합몰을 만든다고 6일 밝혔다.


### 실습
- 주가 알려주기
- 내가 질문한 내용을 파일로 저장하기 -> with open (file.md , w...)
- 아까 질문한 파일을 조회하기
- 그 외 자유롭게

In [ ]:
def stock(item):
    return f"{item}의 오늘 주가는 10% 상승했습니다."

tools = [{
    "type": "function",
    "function": {
        "name": "recommend_clothes",
        "description": "옷에 대해서 물어봤을 때 코멘트를 해준다. 80년대 스타일이라는 말 꼭 넣는다",
        "parameters": {
            "type": "object",
            "properties": {
                "data": {"type": "string", "description": "코멘트 받고 싶은 옷 이름"},
            },
            "required": ["data"],
        },
    },
},
{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "특정 도시의 현재 날씨를 조회한다",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "날씨를 알고 싶은 도시 이름"},
            },
            "required": ["city"],
        },
    },
},
{
    "type": "function",
    "function": {
        "name": "check_data",
        "description": "데이터프레임에서 내용 조회할 때 칼럼명으로 조회해서 알려준다",
        "parameters": {
            "type": "object",
            "properties": {
                "column": {"type": "string", "description": "조회하고 싶은 칼럼"},
            },
            "required": ["column"],
        },
    },
},
{
    "type": "function",
    "function": {
        "name": "stock",
        "description": "질문받은 기업의 주가를 알려준다",
        "parameters": {
            "type": "object",
            "properties": {
                "item": {"type": "string", "description": "주가 알고 싶은 기업"},
            },
            "required": ["item"],
        },
    },
}
]

In [ ]:
# 함수로 만들어서 사용할 수 있음
available_tools = {"get_weather": get_weather, 
                   "recommend_clothes" : recommend_clothes,
                   "check_data" : check_data,
                   "stock" : stock}

def chat_with_tools(question, tools):
    messages = [{"role": "user", "content": question}]
    r = client.chat.completions.create(model="gpt-5.6-luna", tools=tools,
                                       reasoning_effort="none", messages=messages)
    calls = r.choices[0].message.tool_calls
    if not calls:                       # 도구가 필요 없으면 바로 답
        return r.choices[0].message.content
    messages.append(r.choices[0].message)
    for tc in calls:                    # 필요한 도구를 모두 실행
        args = json.loads(tc.function.arguments)
        result = available_tools[tc.function.name](**args)
        messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
    r2 = client.chat.completions.create(model="gpt-5.6-luna", tools=tools,
                                        reasoning_effort="none", messages=messages)
    return r2.choices[0].message.content

한화에어로스페이스는 오늘 **10% 상승**했습니다.


In [67]:
question = "오늘 삼성전자 주가 알려줘"

with open("question.md", "w", encoding="utf-8-sig") as file:
    file.write(question)

print(chat_with_tools(question, tools))

삼성전자는 오늘 **10% 상승**했습니다.


In [ ]:
# openai 기본 내장 툴 사용

messages = [{
    "role" : "user", "content" : "부산 날씨 어때?"
}]

response = client.responses.create(     # 아까의 chat.completion과는 형식이 다르다!
    model="gpt-5.6-luna",
    tools=[{"type" : "web_search"}],
    input=messages

)

In [73]:
print(response.output[2].content[0].text)

부산은 현재 **맑고 약 28°C**예요.  
저녁에는 **24~25°C**, 밤에는 **20~23°C** 정도로 내려가며 대체로 맑겠습니다. 


### OpenAI에서 제공하는 툴
- 내장: web_search, code_interpreter 코드가 잘 실행되는지
- 직접 만든 함수: 사내에서 쓸 수 있도록, 파일 저장 및 수정, 사내 DB 조회 -> 함수로 만들어서 사용
- MCP: 남이 공개한 도구